In [1]:
print("Hello World")

Hello World


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import re

import numpy as np
import pandas as pd
from groq import Groq

In [3]:

ROOT = Path.cwd()
ARTIFACTS_DIR = ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

MODULE4_STRATEGY_PATH = ARTIFACTS_DIR / "module_4_pit_strategy_recommendations.csv"
MODULE3_DRIVER_PATH = ARTIFACTS_DIR / "module_3_driver_performance_predictions.csv"
HISTORICAL_RESULTS_PATH = ROOT / "session_model_data" / "race_results.csv"


In [4]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

In [ ]:
GROQ_API_KEY = "yourapikey"
GROQ_MODEL = "llama-3.3-70b-versatile"

if GROQ_API_KEY.strip():
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY.strip()

if os.getenv("GROQ_API_KEY"):
    print("Groq API key found.")
else:
    print("Groq API key not found yet. Add it before running the LLM generation cell.")

Groq API key found.


In [6]:
DATA_SOURCE = "sample_simulation"  
CUSTOM_JSON_PATH = ROOT / "data" / "module_6_custom_race_result.json"

In [7]:
sample_race_result = {
    "event": {
        "season": 2025,
        "round": 6,
        "race_name": "Simulated Grand Prix",
        "circuit": "Model-generated race scenario",
        "result_type": "simulated",
        "total_laps": 57,
    },
    "race_summary_facts": {
        "winner": "Max Verstappen",
        "podium": ["Max Verstappen", "Lando Norris", "Charles Leclerc"],
        "fastest_lap": "Lando Norris",
        "safety_cars": 1,
        "virtual_safety_cars": 0,
        "main_storyline": "Verstappen converted race pace and pit timing into victory while McLaren used an aggressive two-stop plan with Norris.",
    },
    "results": [
        {"position": 1, "driver": "Max Verstappen", "team": "Red Bull Racing", "grid": 3, "pit_stops": 1, "pit_laps": [24], "overtakes": 8, "fastest_lap": False, "status": "Finished"},
        {"position": 2, "driver": "Lando Norris", "team": "McLaren", "grid": 4, "pit_stops": 2, "pit_laps": [18, 39], "overtakes": 5, "fastest_lap": True, "status": "Finished"},
        {"position": 3, "driver": "Charles Leclerc", "team": "Ferrari", "grid": 2, "pit_stops": 1, "pit_laps": [25], "overtakes": 3, "fastest_lap": False, "status": "Finished"},
        {"position": 4, "driver": "Oscar Piastri", "team": "McLaren", "grid": 1, "pit_stops": 1, "pit_laps": [23], "overtakes": 1, "fastest_lap": False, "status": "Finished"},
        {"position": 5, "driver": "George Russell", "team": "Mercedes", "grid": 5, "pit_stops": 1, "pit_laps": [26], "overtakes": 2, "fastest_lap": False, "status": "Finished"},
        {"position": 6, "driver": "Lewis Hamilton", "team": "Ferrari", "grid": 7, "pit_stops": 1, "pit_laps": [27], "overtakes": 4, "fastest_lap": False, "status": "Finished"},
    ],
    "strategy_notes": [
        "Verstappen used a one-stop strategy and gained track position during the first pit cycle.",
        "Norris used a more aggressive two-stop strategy and had enough tire performance to set fastest lap.",
        "Leclerc protected a podium finish with a conventional one-stop race.",
    ],
    "incidents": [
        {"lap": 12, "description": "Safety car neutralized the field and compressed the pit strategy window."},
        {"lap": 39, "description": "Norris made his second stop and switched to a late attacking stint."},
    ],
}

In [8]:
DRIVER_NAMES = {
    "VER": "Max Verstappen",
    "NOR": "Lando Norris",
    "PIA": "Oscar Piastri",
    "LEC": "Charles Leclerc",
    "SAI": "Carlos Sainz",
    "RUS": "George Russell",
    "HAM": "Lewis Hamilton",
    "ALO": "Fernando Alonso",
    "STR": "Lance Stroll",
    "ALB": "Alex Albon",
    "OCO": "Esteban Ocon",
    "GAS": "Pierre Gasly",
    "TSU": "Yuki Tsunoda",
    "HUL": "Nico Hulkenberg",
    "MAG": "Kevin Magnussen",
    "BOT": "Valtteri Bottas",
    "ZHO": "Zhou Guanyu",
    "LAW": "Liam Lawson",
    "HAD": "Isack Hadjar",
    "ANT": "Andrea Kimi Antonelli",
}


def driver_display_name(code_or_name):
    value = str(code_or_name)
    return DRIVER_NAMES.get(value.upper(), value)


In [ ]:
def load_csv_if_exists(path):
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()


In [10]:
def build_race_result_from_module_artifacts(top_n=10):
    strategy_df = load_csv_if_exists(MODULE4_STRATEGY_PATH)
    driver_df = load_csv_if_exists(MODULE3_DRIVER_PATH)
    historical_df = load_csv_if_exists(HISTORICAL_RESULTS_PATH)

    if strategy_df.empty:
        raise FileNotFoundError(f"Missing Module 4 artifact: {MODULE4_STRATEGY_PATH}")

    race_name = strategy_df["race_name"].iloc[0]
    season = int(strategy_df["season"].iloc[0])
    round_no = int(strategy_df["round"].iloc[0])
    race_laps = int(strategy_df["race_laps"].iloc[0]) if "race_laps" in strategy_df else None

    historical_race = historical_df[
        (historical_df.get("season") == season)
        & (historical_df.get("round") == round_no)
    ] if not historical_df.empty else pd.DataFrame()

    circuit = "Unknown circuit"
    if not historical_race.empty and "circuit" in historical_race.columns:
        circuit = historical_race["circuit"].iloc[0]

    driver_lookup = pd.DataFrame()
    if not driver_df.empty:
        driver_lookup = driver_df[["driver", "predicted_class", "estimated_overtakes", "performance_score"]].copy()

    sim_df = strategy_df.sort_values("predicted_finishing_position").head(top_n).copy()
    if not driver_lookup.empty:
        sim_df = sim_df.merge(driver_lookup, on="driver", how="left")
    else:
        sim_df["predicted_class"] = "Not available"
        sim_df["estimated_overtakes"] = np.nan
        sim_df["performance_score"] = np.nan

    fastest_lap_driver = sim_df.sort_values(
        ["estimated_overtakes", "optimized_rank_score"],
        ascending=[False, True],
        na_position="last",
    )["driver"].iloc[0]

    results = []
    for row in sim_df.itertuples(index=False):
        pit_lap = int(row.suggested_pit_lap) if pd.notna(row.suggested_pit_lap) else None
        results.append({
            "position": int(row.predicted_finishing_position),
            "driver": driver_display_name(row.driver),
            "driver_code": row.driver,
            "team": row.team,
            "grid": int(row.grid_position),
            "pit_stops": 1,
            "pit_laps": [pit_lap] if pit_lap is not None else [],
            "pit_window": [int(row.pit_window_start), int(row.pit_window_end)],
            "overtakes": int(row.estimated_overtakes) if pd.notna(row.estimated_overtakes) else None,
            "driver_class": row.predicted_class,
            "fastest_lap": row.driver == fastest_lap_driver,
            "status": "Simulated finish",
        })

    podium = [item["driver"] for item in sorted(results, key=lambda item: item["position"])[:3]]
    winner = podium[0]
    fastest_lap_name = driver_display_name(fastest_lap_driver)

    return {
        "event": {
            "season": season,
            "round": round_no,
            "race_name": race_name,
            "circuit": circuit,
            "result_type": "simulated from Module 4 strategy output",
            "total_laps": race_laps,
        },
        "race_summary_facts": {
            "winner": winner,
            "podium": podium,
            "fastest_lap": fastest_lap_name,
            "safety_cars": 0,
            "virtual_safety_cars": 0,
            "main_storyline": "Race order is generated from ML finishing-position prediction and pit strategy simulation artifacts.",
        },
        "results": results,
        "strategy_notes": [
            "Pit strategy comes from Module 4 one-stop pit-window recommendations.",
            "Finishing order is sorted by predicted finishing position from the strategy model.",
            "Overtake estimates and driver class are enriched from Module 3 when available.",
        ],
        "incidents": [],
    }

print("Module artifact loader ready.")

Module artifact loader ready.


In [11]:
if DATA_SOURCE == "sample_simulation":
    race_result = sample_race_result
elif DATA_SOURCE == "module_artifacts":
    race_result = build_race_result_from_module_artifacts(top_n=10)
elif DATA_SOURCE == "custom_json":
    race_result = json.loads(CUSTOM_JSON_PATH.read_text(encoding="utf-8"))
else:
    raise ValueError("DATA_SOURCE must be sample_simulation, module_artifacts, or custom_json")

print(json.dumps(race_result, indent=2)[:4000])

{
  "event": {
    "season": 2025,
    "round": 6,
    "race_name": "Simulated Grand Prix",
    "circuit": "Model-generated race scenario",
    "result_type": "simulated",
    "total_laps": 57
  },
  "race_summary_facts": {
    "winner": "Max Verstappen",
    "podium": [
      "Max Verstappen",
      "Lando Norris",
      "Charles Leclerc"
    ],
    "fastest_lap": "Lando Norris",
    "safety_cars": 1,
    "virtual_safety_cars": 0,
    "main_storyline": "Verstappen converted race pace and pit timing into victory while McLaren used an aggressive two-stop plan with Norris."
  },
  "results": [
    {
      "position": 1,
      "driver": "Max Verstappen",
      "team": "Red Bull Racing",
      "grid": 3,
      "pit_stops": 1,
      "pit_laps": [
        24
      ],
      "overtakes": 8,
      "fastest_lap": false,
      "status": "Finished"
    },
    {
      "position": 2,
      "driver": "Lando Norris",
      "team": "McLaren",
      "grid": 4,
      "pit_stops": 2,
      "pit_laps": [
 

In [12]:
results_df = pd.DataFrame(race_result["results"]).sort_values("position").reset_index(drop=True)
results_df["position_change"] = results_df["grid"] - results_df["position"]
results_df["position_change_label"] = np.select(
    [results_df["position_change"] > 0, results_df["position_change"] < 0],
    ["gained " + results_df["position_change"].astype(str), "lost " + results_df["position_change"].abs().astype(str)],
    default="held position",
)
results_df["pit_laps_text"] = results_df["pit_laps"].apply(lambda laps: ", ".join(map(str, laps)) if laps else "not supplied")


In [13]:
team_df = (
    results_df.groupby("team", as_index=False)
    .agg(
        best_finish=("position", "min"),
        drivers=("driver", lambda values: ", ".join(values)),
        avg_finish=("position", "mean"),
        total_overtakes=("overtakes", "sum"),
        net_position_change=("position_change", "sum"),
        pit_stops=("pit_stops", "sum"),
    )
    .sort_values(["best_finish", "avg_finish"], ascending=[True, True])
)

results_df

,position,driver,team,grid,pit_stops,pit_laps,overtakes,fastest_lap,status,position_change,position_change_label,pit_laps_text
0,1,Max Verstappen,Red Bull Racing,3,1,[24],8,False,Finished,2,gained 2,24
1,2,Lando Norris,McLaren,4,2,"[18, 39]",5,True,Finished,2,gained 2,"18, 39"
2,3,Charles Leclerc,Ferrari,2,1,[25],3,False,Finished,-1,lost 1,25
3,4,Oscar Piastri,McLaren,1,1,[23],1,False,Finished,-3,lost 3,23
4,5,George Russell,Mercedes,5,1,[26],2,False,Finished,0,held position,26
5,6,Lewis Hamilton,Ferrari,7,1,[27],4,False,Finished,1,gained 1,27


In [14]:
team_df

,team,best_finish,drivers,avg_finish,total_overtakes,net_position_change,pit_stops
3,Red Bull Racing,1,Max Verstappen,1.0,8,2,1
1,McLaren,2,"Lando Norris, Oscar Piastri",3.0,6,-1,3
0,Ferrari,3,"Charles Leclerc, Lewis Hamilton",4.5,7,0,2
2,Mercedes,5,George Russell,5.0,2,0,1


In [15]:
def build_llm_context(race_result, results_df, team_df):
    event = race_result["event"]
    facts = race_result["race_summary_facts"]

    driver_lines = []
    for row in results_df.itertuples(index=False):
        fastest = " Fastest lap." if bool(row.fastest_lap) else ""
        driver_class = f" Driver class: {row.driver_class}." if hasattr(row, "driver_class") and pd.notna(row.driver_class) else ""
        overtakes = "not supplied" if pd.isna(row.overtakes) else int(row.overtakes)
        driver_lines.append(
            f"P{row.position}: {row.driver} ({row.team}), started P{row.grid}, {row.position_change_label}, "
            f"pit stops: {row.pit_stops}, pit laps: {row.pit_laps_text}, overtakes: {overtakes}.{fastest}{driver_class}"
        )

    team_lines = []
    for row in team_df.itertuples(index=False):
        team_lines.append(
            f"{row.team}: drivers {row.drivers}; best finish P{row.best_finish}; average finish {row.avg_finish:.1f}; "
            f"total overtakes {int(row.total_overtakes) if pd.notna(row.total_overtakes) else 'not supplied'}; "
            f"net position change {int(row.net_position_change)}; total pit stops {int(row.pit_stops)}."
        )

    incident_lines = [f"Lap {item['lap']}: {item['description']}" for item in race_result.get("incidents", [])]

    context = {
        "event": event,
        "headline_facts": facts,
        "driver_results": driver_lines,
        "team_analysis_inputs": team_lines,
        "strategy_notes": race_result.get("strategy_notes", []),
        "incidents": incident_lines,
    }
    return json.dumps(context, indent=2)

llm_context = build_llm_context(race_result, results_df, team_df)
print(llm_context)

{
  "event": {
    "season": 2025,
    "round": 6,
    "race_name": "Simulated Grand Prix",
    "circuit": "Model-generated race scenario",
    "result_type": "simulated",
    "total_laps": 57
  },
  "headline_facts": {
    "winner": "Max Verstappen",
    "podium": [
      "Max Verstappen",
      "Lando Norris",
      "Charles Leclerc"
    ],
    "fastest_lap": "Lando Norris",
    "safety_cars": 1,
    "virtual_safety_cars": 0,
    "main_storyline": "Verstappen converted race pace and pit timing into victory while McLaren used an aggressive two-stop plan with Norris."
  },
  "driver_results": [
    "P1: Max Verstappen (Red Bull Racing), started P3, gained 2, pit stops: 1, pit laps: 24, overtakes: 8.",
    "P2: Lando Norris (McLaren), started P4, gained 2, pit stops: 2, pit laps: 18, 39, overtakes: 5. Fastest lap.",
    "P3: Charles Leclerc (Ferrari), started P2, lost 1, pit stops: 1, pit laps: 25, overtakes: 3.",
    "P4: Oscar Piastri (McLaren), started P1, lost 3, pit stops: 1, pit l

In [16]:
SYSTEM_PROMPT = """
You are a Formula 1 race analyst. Your job is to turn structured race results into a clear post-race report.
Use only the facts supplied in the race context. Do not invent penalties, tire compounds, lap times, radio messages, or weather.
If the race is simulated, describe it as a simulated result. If it is actual, describe it as a completed race.
""".strip()

USER_PROMPT = f"""
Generate an F1 race report from the structured race context below.

Required sections:
1. Race Report
2. Driver Analysis
3. Team Analysis
4. Strategy Summary
5. Key Takeaways

Requirements:
- Mention the winner, podium, fastest lap, overtakes, safety cars, and pit strategies when supplied.
- Explain how strategy affected the result.
- Compare at least three drivers.
- Compare at least three teams if the data includes them.
- Keep the tone like a professional motorsport article.
- Stay grounded in the data.

Structured race context:
{llm_context}
""".strip()

print(USER_PROMPT[:3000])

Generate an F1 race report from the structured race context below.

Required sections:
1. Race Report
2. Driver Analysis
3. Team Analysis
4. Strategy Summary
5. Key Takeaways

Requirements:
- Mention the winner, podium, fastest lap, overtakes, safety cars, and pit strategies when supplied.
- Explain how strategy affected the result.
- Compare at least three drivers.
- Compare at least three teams if the data includes them.
- Keep the tone like a professional motorsport article.
- Stay grounded in the data.

Structured race context:
{
  "event": {
    "season": 2025,
    "round": 6,
    "race_name": "Simulated Grand Prix",
    "circuit": "Model-generated race scenario",
    "result_type": "simulated",
    "total_laps": 57
  },
  "headline_facts": {
    "winner": "Max Verstappen",
    "podium": [
      "Max Verstappen",
      "Lando Norris",
      "Charles Leclerc"
    ],
    "fastest_lap": "Lando Norris",
    "safety_cars": 1,
    "virtual_safety_cars": 0,
    "main_storyline": "Verstap

In [17]:
def generate_race_report(prompt, model=GROQ_MODEL, temperature=0.35, max_tokens=1800):
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("Missing GROQ_API_KEY. Add your Groq API key before running this cell.")

    client = Groq(api_key=api_key)
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return completion.choices[0].message.content

race_report = generate_race_report(USER_PROMPT)
print(race_report)

## 1. Race Report
The Simulated Grand Prix, a 57-lap model-generated race scenario, concluded with Max Verstappen claiming victory. Verstappen, starting from third, executed a flawless one-stop strategy to overtake his competitors and secure the top spot on the podium. Lando Norris, who started fourth, and Charles Leclerc, starting second, completed the podium in second and third place, respectively. Norris also achieved the fastest lap of the race. The event was marked by a single safety car deployment on lap 12, which significantly impacted the pit strategy window for all drivers.

## 2. Driver Analysis
Comparing the performances of Verstappen, Norris, and Leclerc provides insight into the strategic decisions that influenced the outcome. Verstappen's ability to gain two positions, making eight overtakes, and managing his tires effectively on a one-stop strategy, was key to his victory. Norris, despite starting lower than Leclerc, managed to gain two positions and set the fastest lap,

In [18]:
def create_local_preview_report(race_result, results_df, team_df):
    event = race_result["event"]
    facts = race_result["race_summary_facts"]
    winner_row = results_df.iloc[0]
    fastest = facts.get("fastest_lap", "not supplied")
    podium = ", ".join(facts.get("podium", []))

    top_driver_lines = []
    for row in results_df.head(5).itertuples(index=False):
        top_driver_lines.append(
            f"- {row.driver}: P{row.position}, started P{row.grid}, {row.position_change_label}, "
            f"{row.pit_stops} stop(s), {row.overtakes if pd.notna(row.overtakes) else 'unknown'} overtakes."
        )

    team_lines = []
    for row in team_df.head(5).itertuples(index=False):
        team_lines.append(
            f"- {row.team}: best finish P{row.best_finish}, drivers {row.drivers}, total pit stops {int(row.pit_stops)}."
        )

    return f"""# Local Preview: {event['race_name']}

## Race Report
{facts['winner']} won the {event['result_type']} {event['race_name']}. The podium was {podium}. Fastest lap: {fastest}. Safety cars: {facts.get('safety_cars', 'not supplied')}.

## Driver Analysis
{chr(10).join(top_driver_lines)}

## Team Analysis
{chr(10).join(team_lines)}

## Strategy Summary
{winner_row.driver} used {winner_row.pit_stops} stop(s), with pit lap(s): {winner_row.pit_laps_text}. {facts.get('main_storyline', '')}
"""

preview_report = create_local_preview_report(race_result, results_df, team_df)
print(preview_report)

# Local Preview: Simulated Grand Prix

## Race Report
Max Verstappen won the simulated Simulated Grand Prix. The podium was Max Verstappen, Lando Norris, Charles Leclerc. Fastest lap: Lando Norris. Safety cars: 1.

## Driver Analysis
- Max Verstappen: P1, started P3, gained 2, 1 stop(s), 8 overtakes.
- Lando Norris: P2, started P4, gained 2, 2 stop(s), 5 overtakes.
- Charles Leclerc: P3, started P2, lost 1, 1 stop(s), 3 overtakes.
- Oscar Piastri: P4, started P1, lost 3, 1 stop(s), 1 overtakes.
- George Russell: P5, started P5, held position, 1 stop(s), 2 overtakes.

## Team Analysis
- Red Bull Racing: best finish P1, drivers Max Verstappen, total pit stops 1.
- McLaren: best finish P2, drivers Lando Norris, Oscar Piastri, total pit stops 3.
- Ferrari: best finish P3, drivers Charles Leclerc, Lewis Hamilton, total pit stops 2.
- Mercedes: best finish P5, drivers George Russell, total pit stops 1.

## Strategy Summary
Max Verstappen used 1 stop(s), with pit lap(s): 24. Verstappen conver

In [19]:
event = race_result["event"]
raw_name = f"module_6_{event['season']}_{event['race_name']}_{event['result_type']}_race_report"
slug = re.sub(r"[^a-zA-Z0-9]+", "_", raw_name).strip("_").lower()
report_path = ARTIFACTS_DIR / f"{slug}.md"

report_text = globals().get("race_report", preview_report)
report_markdown = f"""# Module 6 Race Summary Generator Output

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Model: {GROQ_MODEL if 'race_report' in globals() else 'local preview, Groq not called'}
Data source: {DATA_SOURCE}
Race: {event['season']} {event['race_name']}
Result type: {event['result_type']}

{report_text}
"""

report_path.write_text(report_markdown, encoding="utf-8")
print(f"Saved report to: {report_path}")

Saved report to: /Users/santhoshkumarv/vs_code_projects/internship-harshith/projects/capstone_project/artifacts/module_6_2025_simulated_grand_prix_simulated_race_report.md
